# Greedy Algorithms

A paradigm: build a solution by repeatedly taking the **locally best** choice and never reconsidering. When it works it's beautifully simple and fast (usually a sort + one pass) — but it **only works when the problem has the greedy-choice property**, and proving that is the real task. We'll see classic wins (activity selection, fractional knapsack) and an instructive failure (coin change, which needs DP).

**Contents**

1. **The signal** — and the catch
2. **Activity selection** / interval scheduling
3. **Fractional knapsack**
4. **Huffman coding** — optimal prefix codes
5. **When greedy fails** — and the exchange argument
6. **When to use & cheat-sheet**

## 1. The signal — and the catch

A **greedy** algorithm makes the choice that looks best *right now* and never backtracks. The appeal is speed and simplicity — typically **sort, then one pass**. The catch is correctness: greedy is right only when the problem has the **greedy-choice property** (a locally optimal choice belongs to *some* global optimum) plus **optimal substructure**.

That's a real condition, not a given. The hard part of a greedy solution is **proving** the greedy choice is safe — usually via an *exchange argument* (§4). When the property fails, greedy returns a wrong answer and you need **dynamic programming** instead (the DP notebook). Rule of thumb: *greedy commits and never reconsiders; DP explores all options.* Use greedy only when you can argue that committing is safe.

## 2. Activity selection / interval scheduling

Given intervals, select the **most** that don't overlap. The greedy choice: always take the interval that **finishes earliest** — it leaves the most room for the rest. Sort by end time, then sweep, taking each interval whose start is ≥ the last chosen end:

In [1]:
def max_activities(intervals):
    chosen, last_end = [], float("-inf")
    for start, end in sorted(intervals, key=lambda iv: iv[1]):   # earliest finish first
        if start >= last_end:
            chosen.append((start, end))
            last_end = end
    return chosen

print("selected:", max_activities([(1, 3), (2, 5), (4, 7), (1, 8), (5, 9), (8, 10)]))


selected: [(1, 3), (4, 7), (8, 10)]


"Earliest finish" is the provably-optimal greedy choice here; "earliest start" or "shortest interval" are tempting but **wrong**. Picking the right greedy criterion is the whole game — and this is the sweep-line family from the patterns tier.

## 3. Fractional knapsack

Maximize value in a capacity-limited knapsack when you can take **fractions** of items. The greedy choice: take items by **value-to-weight ratio**, filling completely until the last item, which you take partially. Sorting by ratio makes it O(n log n):

In [2]:
def fractional_knapsack(items, capacity):    # items: (value, weight)
    total = 0.0
    for value, weight in sorted(items, key=lambda iw: iw[0] / iw[1], reverse=True):
        take = min(weight, capacity)          # all of it, or whatever fits
        total += value * take / weight
        capacity -= take
        if capacity == 0:
            break
    return total

print("fractional_knapsack:", fractional_knapsack([(60, 10), (100, 20), (120, 30)], 50))


fractional_knapsack: 240.0


The **fractional** version is greedy-solvable; the **0/1** knapsack (each item all-or-nothing) is **not** — greedy by ratio can be arbitrarily wrong there, which is exactly why it needs DP (the DP notebook). One word — "fractional" — flips the right tool.

## 4. Huffman coding — optimal prefix codes

Given symbols with frequencies, build a binary code that minimizes total encoded length. The trick is **variable-length, prefix-free** codes: frequent symbols get short codes, rare ones get long codes, and no code is a prefix of another (so the stream decodes unambiguously with no separators).

The greedy choice: repeatedly **merge the two lowest-frequency nodes** into a parent whose frequency is their sum, until one tree remains. A min-heap (**heaps**) gives the two smallest in O(log n) each, so the whole build is O(n log n). Assigning `0` going left and `1` going right down the finished tree yields the codes — leaves (real symbols) sit at the ends, so no symbol's code is a prefix of another's.

Build the tree with `heapq`. Each heap entry is `(freq, tiebreak, node)`; the tiebreak counter keeps comparisons total-ordered (and deterministic) without ever comparing the node payloads. Leaves are `(symbol, None, None)`; internal nodes carry their two children:

In [3]:
import heapq
from collections import Counter
from itertools import count

def build_tree(freqs):                                   # freqs: {symbol: weight}
    tie = count()                                        # stable, payload-free tiebreak
    heap = [(w, next(tie), (sym, None, None)) for sym, w in freqs.items()]
    heapq.heapify(heap)
    while len(heap) > 1:
        w1, _, n1 = heapq.heappop(heap)                  # two lowest frequencies (heaps)
        w2, _, n2 = heapq.heappop(heap)
        heapq.heappush(heap, (w1 + w2, next(tie), (None, n1, n2)))   # merged parent
    return heap[0][2]                                    # the root node

def build_codes(root):
    codes = {}
    def walk(node, prefix):
        sym, left, right = node
        if left is None and right is None:               # a leaf: real symbol
            codes[sym] = prefix or "0"                   # single-symbol edge case
            return
        walk(left, prefix + "0")                         # 0 = go left
        walk(right, prefix + "1")                        # 1 = go right
    walk(root, "")
    return codes

text = "abracadabra"
freqs = Counter(text)
codes = build_codes(build_tree(freqs))
print("freqs:", dict(freqs))
for sym, c in sorted(codes.items(), key=lambda kv: len(kv[1])):
    print(f"  {sym!r}: {c}")


freqs: {'a': 5, 'b': 2, 'r': 2, 'c': 1, 'd': 1}
  'a': 0
  'c': 100
  'd': 101
  'b': 110
  'r': 111


Encoding is a lookup-and-concatenate; decoding walks the tree bit by bit, emitting a symbol each time it reaches a leaf, then restarting at the root:

In [4]:
def encode(text, codes):
    return "".join(codes[ch] for ch in text)

def decode(bits, root):
    out, node = [], root
    for b in bits:
        sym, left, right = node
        node = left if b == "0" else right
        sym, left, right = node
        if left is None and right is None:               # reached a leaf
            out.append(sym)
            node = root                                  # restart for the next symbol
    return "".join(out)

encoded = encode(text, codes)
decoded = decode(encoded, build_tree(freqs))
print("encoded:", encoded)
print("decoded:", decoded)
assert decoded == text                                   # (1) round-trip recovers original
print("round-trip ok:", decoded == text)


encoded: 01101110100010101101110
decoded: abracadabra
round-trip ok: True


**Claim (2): the codes are prefix-free** — no code is a prefix of another, which is what makes decoding unambiguous. We can check it directly, and also confirm Kraft's inequality (sum of 2^-length over the codes equals 1 for a full binary code), the structural fingerprint of a complete prefix code:

In [5]:
from fractions import Fraction

def is_prefix_free(codewords):
    cws = sorted(codewords)
    return all(not b.startswith(a) for a, b in zip(cws, cws[1:]))  # sorted: only neighbors collide

words = list(codes.values())
assert is_prefix_free(words)                             # (2) prefix-free
kraft = sum(Fraction(1, 2 ** len(c)) for c in words)     # exact, no float error
assert kraft == 1                                        # full tree => Kraft sum is exactly 1
print("prefix-free:", is_prefix_free(words), " Kraft sum:", kraft)


prefix-free: True  Kraft sum: 1


**Claim (3): on skewed input Huffman beats the fixed-length code.** A fixed-length code needs `ceil(log2(alphabet))` bits per symbol regardless of frequency; Huffman spends fewer bits on the common symbols. The more skewed the distribution, the bigger the win:

In [6]:
import math, random

random.seed(42)
alphabet = "abcdef"
# heavily skewed: 'a' is ~50x as common as 'f'
weights = [50, 25, 12, 6, 3, 1]
sample = random.choices(alphabet, weights=weights, k=5000)

freqs2 = Counter(sample)
codes2 = build_codes(build_tree(freqs2))
huffman_bits = sum(len(codes2[ch]) for ch in sample)

fixed_width = math.ceil(math.log2(len(alphabet)))        # bits per symbol, fixed-length
fixed_bits = fixed_width * len(sample)

assert huffman_bits < fixed_bits                         # (3) beats fixed-length on skew
print(f"fixed-length: {fixed_width} bits/sym -> {fixed_bits} bits")
print(f"huffman     : {huffman_bits} bits ({huffman_bits / len(sample):.2f} bits/sym)")
print(f"saved       : {100 * (1 - huffman_bits / fixed_bits):.1f}%")


fixed-length: 3 bits/sym -> 15000 bits
huffman     : 9266 bits (1.85 bits/sym)
saved       : 38.2%


**Claim (4): Huffman is optimal** — no prefix code has a smaller total weighted length. On a small alphabet we can *prove* it by brute force: enumerate **every** prefix-free code (every assignment of leaf depths satisfying Kraft, realized as an actual binary tree) and confirm none beats Huffman. We generate candidate codes by building all binary trees with the right number of leaves, then check every assignment of symbols to leaves is no better:

In [7]:
from itertools import permutations

def all_leaf_depths(n):
    """Every multiset of leaf depths for a full binary tree with n leaves."""
    if n == 1:
        yield (0,)
        return
    for k in range(1, n):                                # k leaves left, n-k right
        for ld in all_leaf_depths(k):
            for rd in all_leaf_depths(n - k):
                yield tuple(d + 1 for d in ld) + tuple(d + 1 for d in rd)

def brute_force_optimal(weights):
    """Min total weighted length over ALL prefix codes (any tree shape, any assignment)."""
    best = math.inf
    seen = set()
    for depths in all_leaf_depths(len(weights)):
        depths = tuple(sorted(depths))
        if depths in seen:                               # dedupe identical depth multisets
            continue
        seen.add(depths)
        # best assignment for a fixed depth multiset: heaviest symbol -> shallowest leaf
        for perm in permutations(weights):
            cost = sum(w * d for w, d in zip(perm, depths))
            best = min(best, cost)
    return best

def huffman_cost(freqs):
    tree, codes_local = build_tree(freqs), None
    codes_local = build_codes(tree)
    return sum(freqs[s] * len(codes_local[s]) for s in freqs)

random.seed(7)
for trial in range(200):
    k = random.randint(2, 6)
    fr = {chr(97 + i): random.randint(1, 20) for i in range(k)}
    hf = huffman_cost(fr)
    opt = brute_force_optimal(list(fr.values()))
    assert hf == opt, (fr, hf, opt)                      # (4) Huffman == brute-force optimum
print("200 random alphabets: Huffman == brute-force optimum every time")


200 random alphabets: Huffman == brute-force optimum every time


**Why merging the two smallest is safe (the exchange argument, §6).** In an optimal prefix code the two least-frequent symbols can always be taken to be **siblings at the deepest level**: if a deepest leaf held a heavier symbol, swapping it with one of the two lightest could only lower (never raise) the total weighted length — that's the exchange. So merging the two smallest into one node loses no optimality, and the remaining subproblem (with that merged node as a single symbol of summed weight) has the same structure. Greedy choice + optimal substructure = an optimal code.

## 5. When greedy fails — and the exchange argument

The cautionary tale is **coin change**. With canonical coin systems (like US coins) greedy works — always take the largest coin that fits. For an arbitrary set it can be wrong:

In [8]:
def greedy_coins(coins, amount):
    count = 0
    for c in sorted(coins, reverse=True):
        count += amount // c
        amount %= c
    return count if amount == 0 else -1

print("greedy [1,5,10,25], 30:", greedy_coins([1, 5, 10, 25], 30), "(optimal)")
print("greedy [1,3,4], 6     :", greedy_coins([1, 3, 4], 6), "(WRONG: 4+1+1; DP finds 3+3 = 2)")


greedy [1,5,10,25], 30: 2 (optimal)
greedy [1,3,4], 6     : 3 (WRONG: 4+1+1; DP finds 3+3 = 2)


The `[1,3,4]` case (straight from the DP notebook) shows greedy committing to the 4 and missing 3+3. **Why does greedy work for activity selection but not this?** The proof tool is the **exchange argument**: show any optimal solution can be transformed step by step into the greedy one *without getting worse*, so greedy is at least as good. When you can make that argument (earliest-finish, ratio-sort), greedy is safe; when you can't, it isn't a greedy problem.

## 6. When to use & cheat-sheet

| Problem | Greedy choice | Works? |
|---|---|:---:|
| Activity selection | earliest finish time | ✅ provable |
| Fractional knapsack | highest value/weight ratio | ✅ |
| Huffman coding | merge the two least-frequent | ✅ |
| Dijkstra / Prim / Kruskal | nearest / cheapest edge | ✅ (graph-algorithms) |
| Jump game (reachability) | farthest reach so far | ✅ |
| 0/1 knapsack | by ratio | ❌ → DP |
| Coin change (arbitrary coins) | largest coin | ❌ → DP |

**The decision:** greedy if you can **prove** the greedy-choice property (exchange argument); otherwise **DP** explores the options greedy would wrongly discard. Many greedy algorithms are just a `sorted(...)` plus a one-pass scan — cheap when they're correct. One more, the **jump game** (greedily track the farthest reachable index):

In [9]:
def can_jump(nums):                          # can you reach the last index?
    reach = 0
    for i, n in enumerate(nums):
        if i > reach:                        # this index is unreachable
            return False
        reach = max(reach, i + n)            # greedily extend the farthest reach
    return True

print("can_jump([2,3,1,1,4]):", can_jump([2, 3, 1, 1, 4]))
print("can_jump([3,2,1,0,4]):", can_jump([3, 2, 1, 0, 4]))


can_jump([2,3,1,1,4]): True
can_jump([3,2,1,0,4]): False
